# ModelNet10 Voxel Cache Visualization

This notebook inspects and visualizes voxel caches produced by `scripts/prepare_modelnet_voxels.py` or subtype caches produced by `scripts/build_modelnet_subtypes.py`.

Expected cache format:

- `train_x`: uint8 `[N, D, H, W]`, voxel tokens where `0 = empty`, `1 = occupied`
- `train_y`: int labels remapped to `0..num_labels-1`
- `test_x`, `test_y`
- optional metadata such as `class_names`, `subtype_names`, `resolution`, `filled_interiors`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_repo_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    return path


REPO_ROOT = find_repo_root()
CACHE_PATH = REPO_ROOT / "data" / "modelnet10_voxel_64_top4_subtypes.npz"

print("repo root:", REPO_ROOT)
print("cache path:", CACHE_PATH)

## Load Cache

In [ ]:
if not CACHE_PATH.exists():
    raise FileNotFoundError(
        f"Cache not found: {CACHE_PATH}\n"
        "Create it first with:\n"
        "Create the base cache first with:\n"
        "uv run python scripts/prepare_modelnet_voxels.py "
        "--input data/ModelNet10 "
        "--output data/modelnet10_voxel_64_top4.npz "
        "--resolution 64 "
        "--num-model-classes 4 "
        "--overwrite\n"
        "Then train a voxel classifier and run scripts/build_modelnet_subtypes.py."
    )

cache = np.load(CACHE_PATH, allow_pickle=False)
print("keys:", sorted(cache.files))

for key in ("train_x", "train_y", "test_x", "test_y"):
    if key in cache:
        arr = cache[key]
        print(f"{key:8s} shape={arr.shape} dtype={arr.dtype}")

In [ ]:
def scalar_metadata(key: str):
    if key not in cache:
        return None
    value = cache[key]
    if np.asarray(value).shape == ():
        return np.asarray(value).item()
    return value


class_names = cache["class_names"].astype(str).tolist() if "class_names" in cache else None
subtype_names = cache["subtype_names"].astype(str).tolist() if "subtype_names" in cache else None

print("class_names:", class_names)
print("subtype_names:", subtype_names)
for key in ("resolution", "voxel_token_classes", "num_model_classes", "filled_interiors", "surface_dilation"):
    print(f"{key:20s}:", scalar_metadata(key))

## Dataset Statistics

In [ ]:
def split_arrays(split: str = "train") -> tuple[np.ndarray, np.ndarray]:
    x_key = f"{split}_x"
    y_key = f"{split}_y"
    if x_key not in cache or y_key not in cache:
        raise KeyError(f"Missing {x_key}/{y_key} in cache")
    return cache[x_key], cache[y_key]


def label_name(label: int) -> str:
    if subtype_names is not None and label < len(subtype_names):
        return f"{label}: {subtype_names[label]}"
    if class_names is None or label >= len(class_names):
        return str(label)
    return f"{label}: {class_names[label]}"


for split in ("train", "test"):
    x, y = split_arrays(split)
    occupied = x.reshape(x.shape[0], -1).sum(axis=1)
    print(f"{split}: samples={len(x)}, grid={x.shape[1:]}, occupied mean={occupied.mean():.1f}, min={occupied.min()}, max={occupied.max()}")
    labels, counts = np.unique(y, return_counts=True)
    print("  labels:", {label_name(int(label)): int(count) for label, count in zip(labels, counts)})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)
for ax, split in zip(axes, ("train", "test")):
    _, y = split_arrays(split)
    labels, counts = np.unique(y, return_counts=True)
    names = [label_name(int(label)) for label in labels]
    ax.bar(names, counts)
    ax.set_title(f"{split} label counts")
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel("samples")
plt.show()

## 3D Voxel View

Change `SPLIT` and `INDEX` to inspect a specific item.

In [ ]:
def plot_voxel(voxels: np.ndarray, title: str = "", ax=None, color: str = "#3b82f6"):
    occupied = voxels.astype(bool)
    if ax is None:
        fig = plt.figure(figsize=(5, 5))
        ax = fig.add_subplot(111, projection="3d")
    ax.voxels(occupied, facecolors=color, edgecolor="k", linewidth=0.05)
    ax.set_title(title)
    ax.set_axis_off()
    ax.set_box_aspect(occupied.shape)
    return ax


SPLIT = "train"
INDEX = 0

x, y = split_arrays(SPLIT)
voxels = x[INDEX]
label = int(y[INDEX])
title = f"{SPLIT}[{INDEX}]  {label_name(label)}  occupied={int(voxels.sum())}"
plot_voxel(voxels, title=title)
plt.show()

## Orthogonal Slices

The slice view is useful when dense 3D rendering becomes hard to read.

In [ ]:
def plot_slices(voxels: np.ndarray, title: str = "", indices: tuple[int, int, int] | None = None):
    d, h, w = voxels.shape
    if indices is None:
        indices = (d // 2, h // 2, w // 2)
    z, y, x = indices
    slices = [
        (f"z={z}", voxels[z, :, :]),
        (f"y={y}", voxels[:, y, :]),
        (f"x={x}", voxels[:, :, x]),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
    fig.suptitle(title)
    for ax, (name, image) in zip(axes, slices):
        ax.imshow(image, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(name)
        ax.axis("off")
    plt.show()


plot_slices(voxels, title=title)

## Examples By Label

In [ ]:
def plot_first_example_per_label(split: str = "train"):
    x, y = split_arrays(split)
    labels = np.unique(y)
    fig = plt.figure(figsize=(4 * len(labels), 4))
    for col, label in enumerate(labels, start=1):
        idx = int(np.flatnonzero(y == label)[0])
        ax = fig.add_subplot(1, len(labels), col, projection="3d")
        plot_voxel(x[idx], title=f"{split}[{idx}]\n{label_name(int(label))}", ax=ax)
    plt.tight_layout()
    plt.show()


plot_first_example_per_label("train")

## Inspect The Most / Least Occupied Shapes

In [ ]:
x, y = split_arrays("train")
occupied = x.reshape(x.shape[0], -1).sum(axis=1)
order = np.argsort(occupied)
indices = [int(order[0]), int(order[len(order) // 2]), int(order[-1])]

fig = plt.figure(figsize=(12, 4))
for col, idx in enumerate(indices, start=1):
    ax = fig.add_subplot(1, 3, col, projection="3d")
    title = f"train[{idx}]\n{label_name(int(y[idx]))}\noccupied={int(occupied[idx])}"
    plot_voxel(x[idx], title=title, ax=ax)
plt.tight_layout()
plt.show()